In [ ]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from interpret.glassbox import ExplainableBoostingRegressor
from sklearn.metrics import mean_absolute_error
from interpret import show
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

#### Generating synthetic data from JSON format

In [ ]:
# Data generation
n = 1000
languages = ["French", "English", "Spanish", "German", "Arabic"]
document_types = ["sales_contract", "passport", "birth_certificate", "academic_transcript"]
units = ["page", "word"]
currencies = ["eur", "usd"]

data = []

for _ in range(n):
    src = random.choice(languages)
    tgt = random.choice([lang for lang in languages if lang != src])
    unit = random.choice(units)
    nb_unit = random.randint(1, 50)
    doc = random.choice(document_types)
    delay = random.randint(1, 10)
    cur = random.choice(currencies)

    base_price = 20 * nb_unit
    variation = random.randint(-20, 20)
    proposals = [base_price + variation + offset for offset in [-10, 0, 10]]
    accepted_index = random.randint(0, 2)
    accepted = random.random() > 0.2
    proposed_price_accepted = proposals[accepted_index] if accepted else None

    data.append({
        "source_language": src,
        "target_language": tgt,
        "unit": unit,
        "number_of_unit": nb_unit,
        "document_type": doc,
        "deadline_unit": "days",
        "requested_deadline": delay,
        "currency": cur,
        "proposed_price": proposals,
        "proposed_price_accepted": proposed_price_accepted,
        "price_accepted": accepted
    })

df = pd.DataFrame(data)
df.to_csv("synthetic_quotes.csv", index=False)
df.head()

#### Data preparation for the EBM model

In [ ]:
# Reading and preprocessing
df = pd.read_csv("synthetic_quotes.csv")

# Delete lines without accepted price
df = df[df["proposed_price_accepted"].notnull()].copy()

# Converting categorical columns
for col in ["source_language", "target_language", "unit", "document_type"]:
    df[col] = df[col].astype("category")

# We keep the useful features
df["price"] = df["proposed_price_accepted"]
X = df[[
    "source_language", "target_language",
    "unit",
    "number_of_unit", "document_type",
    "requested_deadline"
]]
y = df["price"]

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### EBM model training + visualization

In [ ]:
ebm = ExplainableBoostingRegressor(random_state=0)
ebm.fit(X_train, y_train)

# Prediction
y_pred = ebm.predict(X_test)
print("MAE :", mean_absolute_error(y_test, y_pred))

# Interactive visualization in the notebook
ebm_global = ebm.explain_global()
for i, name in enumerate(ebm_global.data()['names']):
    print(f"Feature {i}: {name}")
ebm_global.visualize(1)

#### Matplotlib/Seaborn display of importances

In [ ]:
# Extract feature names and flattened scores
features = ebm.feature_names
raw_scores = ebm.term_scores_

# Flattening: we take the average of each term (useful for interactions or non-linear functions)
importances = [np.mean(s) if isinstance(s, (list, np.ndarray)) else s for s in raw_scores]

# Display with seaborn
sns.barplot(x=importances, y=features)
plt.title("Importance moyenne des variables selon EBM")
plt.xlabel("Contribution moyenne")
plt.tight_layout()
plt.show()

In [ ]:
# Example of a simulated request as in TIPS
new_request = {
    "source_language": "French",
    "target_language": "English",
    "unit": "page",
    "number_of_unit": 12,
    "document_type": "sales_contract",
    "requested_deadline": 3
}

features_to_use = ["source_language", "target_language", "unit", "number_of_unit", "document_type", "requested_deadline"]

X_new = pd.DataFrame([new_request])[features_to_use]

# Prediction
predicted_price = ebm.predict(X_new)[0]
print(f"Prix estimé : {predicted_price:.2f} €")


In [ ]:
print(features_to_use)
print(importances)